# C2 - WESAD STRESS FROM HEART SIGNALS (step 10)

**Status: PRELIMINARY, public data. Estimator for one latent state, not the capacity label.**

Question: when a person is under stress, how do heart rate and heart rate variability change compared with the same person's calm periods, and can a small model tell stress from non-stress using only signals a watch or phone-linked wearable can give?

Data: WESAD (Schmidt et al., ICMI 2018), 15 people, lab protocol: baseline, stress (Trier Social Stress Test), amusement, meditation. Only the **wrist device (Empatica E4)** signals are used, because that is what a wearable can offer. Chest signals are not used. WESAD is used on its own and is never merged with StudentLife or SWELL-KW.

Rules fixed before looking at results:
1. People are never split between train and test (participant-grouped cross-validation). The participant code is never a feature.
2. Windows: 60 seconds, 30 second step. Labels come from the dataset protocol, not invented. Windows that mix conditions are dropped.
3. Main target: **stress (1) vs non-stress (0: baseline, amusement, meditation)**. A second target, baseline vs stress, is shown for comparison.
4. Feature sets: **HR/HRV only** (what Health Connect-type data can give), **HR/HRV + movement**, and **+ EDA and skin temperature** (lab ceiling, most phones and watches do not provide EDA).
5. Features are shown **raw** and **person-normalised** (each person's own median and spread across their windows, no labels used). This matches the personal-baseline idea.
6. Shuffled-label control for every model. "Clearly above chance" only if `mean AUC > max(shuffled mean, 0.5) + 2 x shuffled std`.
7. Report accuracy, precision, recall, F1, balanced accuracy, AUC and the confusion matrix, never accuracy alone.
8. This is a lab stress task. It is not everyday stress and it is not a medical measure. Wrist photoplethysmography HRV is noisy during movement.

## Cell 1 - Where the data are
Local run: `data/wesad/WESAD.zip` (downloaded from the dataset page, Uni Siegen, about 2.25 GB). In Colab the cell downloads it to the runtime. Only the per-subject `.pkl` files are extracted, each into its own folder.

In [ ]:
import os, sys, zipfile, pickle, io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    BASE = Path('/content/wesad'); BASE.mkdir(exist_ok=True)
    ZIP = BASE / 'WESAD.zip'
    if not ZIP.exists():
        os.system(f'wget -q -O {ZIP} "https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"')
    OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
    from google.colab import drive
    drive.mount('/content/drive')
else:
    BASE = Path(os.environ.get('C2_WESAD_DIR', 'data/wesad'))
    ZIP = BASE / 'WESAD.zip'
    OUT_DIR = Path(os.environ.get('C2_OUT_DIR', 'outputs'))
OUT_DIR.mkdir(exist_ok=True, parents=True)
EXTRACT = BASE / 'extracted'; EXTRACT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist() if n.endswith('.pkl')]
    for n in names:
        dest = EXTRACT / Path(n).name
        if not dest.exists():
            with z.open(n) as src, open(dest, 'wb') as dst:
                dst.write(src.read())
print('subject files:', sorted(p.name for p in EXTRACT.glob('S*.pkl')))

## Cell 2 - Safe loading and a first look
The `.pkl` files are Python pickles. Pickles can run code, so they are opened with a loader that only allows numpy arrays and plain containers.

In [ ]:
ALLOWED = {
    ('numpy.core.multiarray', '_reconstruct'), ('numpy._core.multiarray', '_reconstruct'),
    ('numpy', 'ndarray'), ('numpy', 'dtype'),
    ('numpy.core.multiarray', 'scalar'), ('numpy._core.multiarray', 'scalar'),
    ('_codecs', 'encode'), ('collections', 'OrderedDict'),
}
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if (module, name) in ALLOWED:
            return super().find_class(module, name)
        raise pickle.UnpicklingError(f'blocked: {module}.{name}')

def load_subject(path):
    with open(path, 'rb') as f:
        d = SafeUnpickler(f, encoding='latin1').load()
    w = d['signal']['wrist']
    return {'bvp': np.asarray(w['BVP']).ravel(), 'eda': np.asarray(w['EDA']).ravel(), 'temp': np.asarray(w['TEMP']).ravel(),
            'acc': np.asarray(w['ACC']), 'label': np.asarray(d['label']).ravel()}

first = sorted(EXTRACT.glob('S*.pkl'))[0]
s0 = load_subject(first)
print(first.name, {k: v.shape for k, v in s0.items()})
print('label counts (700 Hz samples):', dict(zip(*np.unique(s0['label'], return_counts=True))))
print('minutes recorded:', round(len(s0['label']) / 700 / 60, 1))
print('Label meaning: 1 baseline, 2 stress, 3 amusement, 4 meditation, others (0, 5, 6, 7) are transitions or unused and are dropped.')

## Cell 3 - Windows and features
BVP (64 Hz) is band-pass filtered, beats are found, and inter-beat intervals outside 0.33 to 1.5 seconds are discarded. A window needs at least 25 valid intervals, otherwise it is dropped (usually movement). Heart rate and HRV (`rmssd`, `sdnn`, `pnn50`) come from those intervals. Movement is the wrist accelerometer magnitude. EDA and skin temperature are for the lab-ceiling comparison only.

In [ ]:
from scipy.signal import butter, filtfilt, find_peaks
FS = {'bvp': 64, 'acc': 32, 'eda': 4, 'temp': 4, 'label': 700}
WIN, STEP = 60, 30

def bvp_ibi(x):
    b, a = butter(3, [0.5 / 32, 8 / 32], btype='band')
    y = filtfilt(b, a, x)
    pk, _ = find_peaks(y, distance=int(64 * 0.33), prominence=0.4 * np.std(y))
    t = pk / 64.0
    ibi = np.diff(t)
    ibi = ibi[(ibi >= 0.33) & (ibi <= 1.5)]
    return ibi

def window_features(sub, sid):
    n_sec = int(len(sub['label']) / FS['label'])
    rows = []
    for start in range(0, n_sec - WIN + 1, STEP):
        lab = sub['label'][start * 700:(start + WIN) * 700]
        vals, cnt = np.unique(lab, return_counts=True)
        top, frac = vals[np.argmax(cnt)], cnt.max() / len(lab)
        if top not in (1, 2, 3, 4) or frac < 0.95:
            continue
        ibi = bvp_ibi(sub['bvp'][start * 64:(start + WIN) * 64])
        if len(ibi) < 25:
            continue
        d = np.diff(ibi)
        acc = sub['acc'][start * 32:(start + WIN) * 32] / 64.0
        mag = np.sqrt((acc ** 2).sum(axis=1))
        eda = sub['eda'][start * 4:(start + WIN) * 4]
        tmp = sub['temp'][start * 4:(start + WIN) * 4]
        rows.append({'subject': sid, 'start_s': start, 'cond': int(top),
                     'hr_mean': 60.0 / ibi.mean(), 'hr_std': np.std(60.0 / ibi), 'rmssd': np.sqrt(np.mean(d ** 2)) * 1000,
                     'sdnn': ibi.std() * 1000, 'pnn50': np.mean(np.abs(d) > 0.05) * 100,
                     'acc_mean': mag.mean(), 'acc_std': mag.std(),
                     'eda_mean': eda.mean(), 'eda_slope': np.polyfit(np.arange(len(eda)), eda, 1)[0], 'temp_mean': tmp.mean()})
    return rows

ALL = []
for p in sorted(EXTRACT.glob('S*.pkl')):
    sid = p.stem
    sub = load_subject(p)
    r = window_features(sub, sid)
    ALL += r
    print(sid, 'windows kept:', len(r))
    del sub
W = pd.DataFrame(ALL)
W['y_stress'] = (W['cond'] == 2).astype(int)
W.to_csv(OUT_DIR / 'wesad_wrist_windows.csv', index=False)
print('\ntotal windows:', len(W), '| people:', W['subject'].nunique())
print('windows per condition (1 base, 2 stress, 3 amusement, 4 meditation):')
print(W['cond'].value_counts().sort_index().to_string())

## Cell 4 - What changes with stress (the question in plain terms)
Each person's values are expressed against **their own** typical level (median and spread over all their windows), then averaged by condition. Positive = above that person's usual.

In [ ]:
HRV = ['hr_mean', 'hr_std', 'rmssd', 'sdnn', 'pnn50']
MOVE = ['acc_mean', 'acc_std']
LAB = ['eda_mean', 'eda_slope', 'temp_mean']
ALLF = HRV + MOVE + LAB

def person_norm(df, cols):
    out = df.copy()
    for c in cols:
        g = df.groupby('subject')[c]
        med = g.transform('median'); sp = g.transform(lambda s: max(s.quantile(0.75) - s.quantile(0.25), 1e-6))
        out[c] = (df[c] - med) / sp
    return out
WN = person_norm(W, ALLF)
cond_names = {1: 'baseline', 2: 'stress', 3: 'amusement', 4: 'meditation'}
tab = WN.groupby('cond')[['hr_mean', 'rmssd', 'sdnn', 'pnn50', 'acc_mean', 'eda_mean']].mean().rename(index=cond_names)
print('Mean person-normalised value per condition (0 = that person\'s usual):')
print(tab.round(2).to_string())
raw = W.groupby('cond')[['hr_mean', 'rmssd', 'sdnn']].mean().rename(index=cond_names)
print('\nRaw means per condition (bpm / ms):')
print(raw.round(1).to_string())

## Cell 5 - Cross-validation helper
Participant-grouped, 10 different fold layouts, out-of-fold predictions pooled per layout, shuffled-label control.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

def make_model(name):
    if name == 'Majority':
        return DummyClassifier(strategy='prior')
    if name == 'LR':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=150, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1))])

REPEATS = 10
def run_cv(data, cols, label, model_name, shuffled=False):
    X, y, g = data[cols].to_numpy(), data[label].to_numpy(), data['subject'].to_numpy()
    rows, cms = [], []
    for s in range(REPEATS):
        yy = np.random.RandomState(900 + s).permutation(y) if shuffled else y
        prob = np.full(len(yy), np.nan)
        for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s).split(X, yy, g):
            if len(np.unique(yy[tr])) < 2:
                prob[te] = yy[tr].mean(); continue
            prob[te] = make_model(model_name).fit(X[tr], yy[tr]).predict_proba(X[te])[:, 1]
        pred = (prob >= 0.5).astype(int)
        auc = 0.5 if model_name == 'Majority' else roc_auc_score(yy, prob)
        rows.append([accuracy_score(yy, pred), precision_score(yy, pred, zero_division=0), recall_score(yy, pred, zero_division=0),
                     f1_score(yy, pred, zero_division=0), balanced_accuracy_score(yy, pred), auc])
        cms.append(confusion_matrix(yy, pred, labels=[0, 1]))
    return np.array(rows), np.mean(cms, axis=0)

RES, CMS = [], {}
def evaluate(exp, data, cols, label, model_name):
    real, cm = run_cv(data, cols, label, model_name)
    shuf = real if model_name == 'Majority' else run_cv(data, cols, label, model_name, shuffled=True)[0]
    thr = max(shuf[:, 5].mean(), 0.5) + 2 * shuf[:, 5].std()
    RES.append({'experiment': exp, 'model': model_name, 'rows': len(data), 'people': data['subject'].nunique(),
                'accuracy': real[:, 0].mean(), 'precision': real[:, 1].mean(), 'recall': real[:, 2].mean(), 'F1': real[:, 3].mean(),
                'balanced acc': real[:, 4].mean(), 'AUC': real[:, 5].mean(), 'AUC std': real[:, 5].std(), 'shuffled AUC': shuf[:, 5].mean(),
                'clearly above chance?': 'n/a' if model_name == 'Majority' else ('YES' if real[:, 5].mean() > thr else 'no')})
    CMS[(exp, model_name)] = cm
print('helper ready')

## Cell 6 - Experiments
Main target: stress vs non-stress. Secondary: baseline vs stress only.

In [ ]:
SETS = [
    ('S1 HR/HRV, raw', W, HRV),
    ('S2 HR/HRV, person-normalised', WN, HRV),
    ('S3 HR/HRV + movement, person-normalised', WN, HRV + MOVE),
    ('S4 + EDA and temperature (lab ceiling), person-normalised', WN, ALLF),
]
for name, data, cols in SETS:
    for m in ['Majority', 'LR', 'RF']:
        evaluate(name, data, cols, 'y_stress', m)
bs = WN[WN['cond'].isin([1, 2])].copy()
for m in ['Majority', 'LR', 'RF']:
    evaluate('S5 baseline vs stress only, HR/HRV person-normalised', bs, HRV, 'y_stress', m)
res = pd.DataFrame(RES)
show = ['experiment', 'model', 'rows', 'people', 'accuracy', 'precision', 'recall', 'F1', 'balanced acc', 'AUC', 'shuffled AUC', 'clearly above chance?']
print(res[show].round(3).to_string(index=False))
res.round(4).to_csv(OUT_DIR / 'wesad_stress_results.csv', index=False)

## Cell 7 - Confusion matrix and figure
Rows = true class, columns = predicted (0 = non-stress, 1 = stress). Counts are windows, averaged over the 10 layouts.

In [ ]:
import matplotlib;
import matplotlib.pyplot as plt
main = res[(res['experiment'].str.startswith('S2')) & (res['model'] != 'Majority')].sort_values('AUC', ascending=False).iloc[0]
cm = CMS[(main['experiment'], main['model'])]
print(f"{main['experiment']} / {main['model']}: AUC {main['AUC']:.2f}, balanced acc {main['balanced acc']:.2f}, F1 {main['F1']:.2f}")
print(pd.DataFrame(cm.round(0), index=['true non-stress', 'true stress'], columns=['pred non-stress', 'pred stress']).to_string())

rf = res[res['model'] == 'RF'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 3.8)); y = np.arange(len(rf))
ax.barh(y, rf['AUC'], xerr=rf['AUC std'], color='tab:blue', alpha=0.85, label='real labels')
ax.scatter(rf['shuffled AUC'], y, color='k', zorder=3, label='shuffled labels')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(y); ax.set_yticklabels(rf['experiment'], fontsize=8); ax.invert_yaxis()
ax.set_xlim(0.3, 1.0); ax.set_xlabel('ROC-AUC (Random Forest, people held out)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / 'wesad_stress_results.png', dpi=120); plt.show()

## Cell 8 - Direction of the effect
Logistic Regression on the person-normalised HR/HRV features (fitted on everyone, a description only). Positive = pushes towards stress.

In [ ]:
lr = make_model('LR').fit(WN[HRV], WN['y_stress'])
coef = pd.Series(lr.named_steps['clf'].coef_[0], index=HRV).sort_values()
print(coef.round(2).to_string())

## Cell 9 - How to read this (report text)
- Compare **S2 with S1**: if person-normalised beats raw, the personal-baseline idea is supported on these data.
- Compare **S2 with S3**: does adding movement help, or is HR/HRV alone enough?
- **S4 is a ceiling**, not a deployable result, because phones and most watches give no EDA.
- A model counts only if `clearly above chance? = YES` and it beats the Majority row.
- **Limits to state**: 15 people, lab stress task, Empatica E4 wrist signals, short windows, no everyday life. The output is a **physiological stress load**, not a diagnosis.